In [0]:
# ============================================================
# PIPELINE DRIVER — runs generation -> silver -> gold in sequence
# ============================================================
import time
from datetime import datetime

pipeline_start = time.time()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"=== Pipeline run started: {run_id} ===")

results = []

def run_stage(stage_name, notebook_path, timeout=1200):
    stage_start = time.time()
    print(f"\n--- Running: {stage_name} ---")
    try:
        result = dbutils.notebook.run(notebook_path, timeout_seconds=timeout)
        duration = round(time.time() - stage_start, 1)
        print(f"✔ {stage_name} completed in {duration}s")
        results.append({"stage": stage_name, "status": "SUCCESS", "duration_sec": duration})
        return result
    except Exception as e:
        duration = round(time.time() - stage_start, 1)
        print(f"✘ {stage_name} FAILED after {duration}s: {str(e)}")
        results.append({"stage": stage_name, "status": "FAILED", "duration_sec": duration, "error": str(e)})
        raise  # stop the pipeline — don't run silver/gold on broken bronze data

In [0]:
# ============================================================
# STAGE 1: DATA GENERATION  
# ============================================================
run_stage(
    "Data Generation",
    "../01_data_generation/generate_synthetic_data"
)

In [0]:
# ============================================================
# STAGE 2:  bronze 
# ============================================================
run_stage(
    "Broze Layer",
    "../02_bronze/03_Bronze"
)

In [0]:
# ============================================================
# STAGE 2: Silver
# ============================================================
run_stage(
    "Silver Transform",
    "../03_silver/03_silver-clean_transform"
)

In [0]:
# ============================================================
# STAGE 3: GOLD AGGREGATION
# ============================================================
run_stage(
    "Gold Aggregation",
    "../04_gold/04_gold-aggregate_metrics"
)

In [0]:
# ============================================================
# PIPELINE SUMMARY
# ============================================================
total_duration = round(time.time() - pipeline_start, 1)

print(f"\n=== Pipeline run {run_id} complete in {total_duration}s ===")
for r in results:
    status_icon = "✔" if r["status"] == "SUCCESS" else "✘"
    print(f"{status_icon} {r['stage']}: {r['status']} ({r['duration_sec']}s)")

# Optional: log run metadata to a table for pipeline observability
import pandas as pd

df_log = spark.createDataFrame(
    pd.DataFrame([{
        "run_id": run_id,
        "run_timestamp": datetime.now(),
        "total_duration_sec": total_duration,
        "stages_completed": len(results),
        "all_succeeded": all(r["status"] == "SUCCESS" for r in results),
    }])
)

df_log.write.format("delta").mode("append").saveAsTable("workspace.default.pipeline_run_log")